# **ETL of Our World in Data: CO<sub>2</sub> Emissions**

## Objectives
, Population and Energy Sources.
* clean the data, apply the iso3 country code

## Inputs

* data/raw/co-emissions-per-capita/co-emissions-per-capita.csv

## Outputs

* data/processed/01a_df_owid_co2_country.csv
* data/processed/01a_df_owid_co2_world.csv



---

# Set Up directories and Import Necessary Libraries

In [2]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [3]:
# Get the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid') # sets a white background with grid lines 

# 1.0 OWID Data

blah blah

# 2.0 OWID CO<sub>2</sub> Emissions Data

In [4]:
data_columns = ["country", "country_code", "year", "co2_pc"]
column_dtypes = {
    "country": str, "country_code": str, "year": int, "co2_pc" : float}

In [5]:
file_dir_name = "/co-emissions-per-capita/co-emissions-per-capita.csv"

df_owid_co2 = pd.read_csv(
    f"{raw_dir}/{file_dir_name}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )

print(f"Total number of rows: {len(df_owid_co2)}")

Total number of rows: 26509


In [6]:
df_owid_co2

,country,country_code,year,co2_pc
0,Afghanistan,AFG,1949,0.001992
1,Afghanistan,AFG,1950,0.010837
2,Afghanistan,AFG,1951,0.011625
3,Afghanistan,AFG,1952,0.011468
4,Afghanistan,AFG,1953,0.013123
...,...,...,...,...
26504,Zimbabwe,ZWE,2020,0.546847
26505,Zimbabwe,ZWE,2021,0.647125
26506,Zimbabwe,ZWE,2022,0.761205
26507,Zimbabwe,ZWE,2023,0.822681


In [7]:
unique_codes = df_owid_co2['country_code'].dropna().unique()
print(f"Total unique country codes: {len(unique_codes)}")

Total unique country codes: 215


In [8]:
import country_converter as coco
cc = coco.CountryConverter()

df_owid_co2["country_code_iso3"] = cc.pandas_convert(
    series=df_owid_co2['country_code'],
    to='ISO3'
)


nan not found in ISO3
OWID_KOS not found in regex
OWID_WRL not found in regex


📌 Lets look into these rows "OWID_KOS", "OWID_WRL" and missing values in the following sections.

## 2.1 Country Code "OWID_KOS"

📌 The `country_code` column in the OWID data with the value `OWID_KOS` refers to the country Kosovo. 

Kosovo is not yet a member of the United Nations and as such the Country_Converter package [on Kosovo](https://github.com/vincentarelbundock/countrycode/issues/305) does know what ISO3 code to assign to it.

Many commercial datasets and the World's bank has provisionally used `XKX` for Kosovo. We will manually use that as the country_code instead.

In [9]:
kos_rows = df_owid_co2[df_owid_co2['country_code'] == 'OWID_KOS']

kos_rows.head(5)

,country,country_code,year,co2_pc,country_code_iso3
13088,Kosovo,OWID_KOS,1994,0.0,not found
13089,Kosovo,OWID_KOS,1995,0.0,not found
13090,Kosovo,OWID_KOS,1996,0.0,not found
13091,Kosovo,OWID_KOS,1997,0.0,not found
13092,Kosovo,OWID_KOS,1998,0.0,not found


In [10]:
# find the rows with country_code = "OWID_KOS" and assign the value "XKX" to the country_code_iso3

df_owid_co2["country_code_iso3"] = df_owid_co2["country_code"].replace("OWID_KOS", "XKX")


kos_rows = df_owid_co2[df_owid_co2['country_code'] == 'OWID_KOS']
kos_rows.head(5)

,country,country_code,year,co2_pc,country_code_iso3
13088,Kosovo,OWID_KOS,1994,0.0,XKX
13089,Kosovo,OWID_KOS,1995,0.0,XKX
13090,Kosovo,OWID_KOS,1996,0.0,XKX
13091,Kosovo,OWID_KOS,1997,0.0,XKX
13092,Kosovo,OWID_KOS,1998,0.0,XKX


## 2.2 Country Code "OWID_WRL" and nan

📌 The `country_code` column in the OWID data with the value `WRL` refers to the aggregation of CO<sub>2</sub> emissions at the World Level. 

This suggests that the dataset might contain aggregations other levels such as at continents levels which could explain the nan values 

In [11]:
wrl_rows = df_owid_co2[df_owid_co2['country_code'] == 'OWID_WRL']

wrl_rows.head(5)

,country,country_code,year,co2_pc,country_code_iso3
26007,World,OWID_WRL,1750,0.012354,OWID_WRL
26008,World,OWID_WRL,1760,0.013339,OWID_WRL
26009,World,OWID_WRL,1770,0.015819,OWID_WRL
26010,World,OWID_WRL,1780,0.017625,OWID_WRL
26011,World,OWID_WRL,1790,0.021648,OWID_WRL


In [12]:
# find the missing values in country_code
nan_rows = df_owid_co2[df_owid_co2['country_code'].isna()]

nan_rows['country'].value_counts()

country
Asia                             230
Asia (excl. China and India)     230
Europe                           230
Europe (excl. EU-27)             230
Europe (excl. EU-28)             230
European Union (28)              230
High-income countries            230
Oceania                          230
North America                    226
North America (excl. USA)        226
European Union (27)              225
South America                    181
Upper-middle-income countries    175
Lower-middle-income countries    173
Africa                           141
Low-income countries             116
Name: count, dtype: int64

In [13]:
nan_rows.head(10)

,country,country_code,year,co2_pc,country_code_iso3
76,Africa,NaN,1884,0.000168,NaN
77,Africa,NaN,1885,0.000279,NaN
78,Africa,NaN,1886,0.000360,NaN
79,Africa,NaN,1887,0.000358,NaN
80,Africa,NaN,1888,0.000603,NaN
81,Africa,NaN,1889,0.000982,NaN
82,Africa,NaN,1890,0.002200,NaN
83,Africa,NaN,1891,0.002192,NaN
84,Africa,NaN,1892,0.003535,NaN
85,Africa,NaN,1893,0.013220,NaN


📌 This confirms the suspicions the the country_code with missing values or with the value `OWID_WRL` are aggregations rows. 

We will extract these and put into a separate df as we want the data granularity at country-year for our main dataset.

In [14]:
# create a  query mask for identifying rows with country_code == "OWID_WRL" or is nan
exclude_mask = (df_owid_co2["country_code"] == "OWID_WRL") | (df_owid_co2["country_code"].isna())

### 2.2.1 Get and Check the World aggregation data

In [15]:
# get a copy of the df_owid_co2 by including all the unwanted rows
df_owid_co2_world = df_owid_co2[exclude_mask].copy()

In [16]:
print(f"OWID_WRL rows: {(df_owid_co2_world["country_code"] == "OWID_WRL").sum()}")
print(f"NaN rows: {df_owid_co2_world["country_code"].isna().sum()}")
print(f"Total rows: {len(df_owid_co2_world)}")

OWID_WRL rows: 230
NaN rows: 3303
Total rows: 3533


📌 Export to csv

In [17]:
df_owid_co2_world.to_csv(f'{processed_dir}/01a_df_owid_co2_world.csv', index=False)
print(f"Exported: {processed_dir}/01a_df_owid_co2_world.csv")

Exported: ../data/processed/01a_df_owid_co2_world.csv


### 2.2.2 Get and Check the country-year dataset

In [18]:
# get a copy of the df_owid_co2 by excluding all the unwanted rows
df_owid_co2_country = df_owid_co2[~exclude_mask].copy()

In [19]:
print(f"Rows with OWID_WRL: {(df_owid_co2_country["country_code"] == "OWID_WRL").sum()}")
print(f"Rows with missing value: {df_owid_co2_country["country_code"].isna().sum()}")

Rows with OWID_WRL: 0
Rows with missing value: 0


In [20]:
df_owid_co2_country["country_name_iso3"] = cc.pandas_convert(
    series=df_owid_co2_country['country_code_iso3'],
    to='name_short'
)

In [21]:
check_kosovo = df_owid_co2_country[df_owid_co2_country["country"] == "Kosovo"]

check_kosovo.head(2)

,country,country_code,year,co2_pc,country_code_iso3,country_name_iso3
13088,Kosovo,OWID_KOS,1994,0.0,XKX,Kosovo
13089,Kosovo,OWID_KOS,1995,0.0,XKX,Kosovo


# 2.3 EDA of the Country's CO<sub>2</sub> Data

In [22]:
df_owid_co2_country["year"].describe()

count    22976.000000
mean      1958.629091
std         49.384825
min       1750.000000
25%       1929.000000
50%       1969.000000
75%       1998.000000
max       2024.000000
Name: year, dtype: float64

📌 The CO<sub>2</sub> starts from 1750 to 2024.

For this project, we will take data from 1950 onwards.

In [28]:
df_owid_co2_country_aft1949 = df_owid_co2_country[df_owid_co2_country["year"] >= 1950]

df_owid_co2_country_b41949 = df_owid_co2_country[df_owid_co2_country["year"] < 1950]

In [29]:
print(f"Number of unaccounted rows: {len(df_owid_co2_country) - len(df_owid_co2_country_aft1949) - len(df_owid_co2_country_b41949)}")

Number of unaccounted rows: 0


In [30]:
df_owid_co2_country_aft1949["year"].describe()

count    15218.000000
mean      1988.165725
std         21.453683
min       1950.000000
25%       1970.000000
50%       1989.000000
75%       2007.000000
max       2024.000000
Name: year, dtype: float64

# 2.4 Export the CO<sub>2</sub> Data to CSV

📌 Export to csv

In [31]:
df_owid_co2_country_aft1949.to_csv(f'{processed_dir}/01a_df_owid_co2_country_aft1949.csv', index=False)
print(f"Exported: {processed_dir}/01a_df_owid_co2_country_aft1949.csv")

Exported: ../data/processed/01a_df_owid_co2_country_aft1949.csv


---